# **Data** **Preparation**

**Inputs (folder `data/`):**
- `activity_sample.parquet`  
- `fork_pr_sample.parquet`  

**Outputs (folder `data/`):**
- `fork_conversion_labels_obs30d.parquet` / `fork_conversion_labels_obs90d.parquet` — only label columns (keys, window length, fork counts, conversion counts and rate, PR list size) for 30- and 90-day observation windows.  
- `modeling_dataset_obs30d.parquet` / `modeling_dataset_obs90d.parquet` — full modeling table: activity features (`log1p_*`, raw/winsor counts), optional stratum helpers, and the same labels as above, one row per repo×season.  

Train/validation/test splits and the final choice of 30- vs 90-day labels belong to the Modeling stage, not this notebook.


## Field reference

### `activity_sample`

- `repo_name` — upstream repo id (`owner/name`); used with `season` as the join key.  
- `season` — `winter_spring` or `summer_autumn` (anchor months from the extract).  
- `repo_url`, `public` — metadata; we drop them here (`public` has no variation in EDA; URL is not a numeric feature).  
- `watches`, `pushes`, `issues`, `issue_comments`, `releases` — event counts; base for winsor, `log1p`, and modeling.

Row grain: one row = one *repository × season*.

### `fork_pr_sample` 

- `repo_name`, `season` — same keys as `activity_sample` (one-to-one).  
- `forks` — list of fork events (user, time).  
- `prs` — list of PR events; SQL uses a long enough window so we can test *PR after fork, within d days* in Python.

### Label files we build: `fork_conversion_labels_obs{30,90}d.parquet`

- `observation_window_days` — 30 or 90.  
- `n_forks` — number of fork events in the row.  
- `n_converted_forks` — forks with at least one PR by the same user in `(fork_time, fork_time + window]`.  
- `fork_conversion_rate` — `n_converted_forks / n_forks` (per-fork definition; not the same as total PRs / total forks from EDA).  
- `n_pr_events` — `len(prs)` (diagnostic).

### Final file: `modeling_dataset_obs{30,90}d.parquet`

- **IDs:** `repo_name`, `repo_id`, `season`  
- **Features:** all `log1p_*` (from raw and from winsorized counts; Modeling can choose a subset)  
- **Optional helpers for splits:** `activity_strength`, `stratum_activity_quintile`, `stratum_fork_quintile`  
- **Labels:** `observation_window_days`, `n_forks`, `n_converted_forks`, `fork_conversion_rate`, `n_pr_events`, `has_conversion`  
- **Raw / winsor** columns are kept for checks and ablations.


## **Select Data**


### Load Data


In [12]:
import numpy as np
import pandas as pd

activity = pd.read_parquet('data/activity_sample.parquet')
fork_pr = pd.read_parquet('data/fork_pr_sample.parquet')


### Non-modeling columns

From EDA, `public` is not informative and `repo_url` is not used as a numeric feature. We remove them to form `prep_activity` for the rest of the pipeline.


In [13]:
_drop = [c for c in ("public", "repo_url") if c in activity.columns]
prep_activity = activity.drop(columns=_drop, errors="ignore").copy()
print("Dropped:", _drop)
print("prep_activity columns:", prep_activity.columns.tolist())


Dropped: ['public', 'repo_url']
prep_activity columns: ['repo_name', 'season', 'watches', 'pushes', 'issues', 'issue_comments', 'releases']


## **Clean Data**


### Winsorize activity counts

Activity metrics are heavy-tailed. We cap each of the five counters at the 99th percentile and store the result in `*_winsor` columns. Raw counts are kept. We do not cap fork/PR event lists: labels use the full event timestamps from `fork_pr_sample`.

Constants used in later steps are defined here as well: activity column names, join keys, and observation window lengths in days.


In [14]:
ACTIVITY_COUNT_COLS = ["watches", "pushes", "issues", "issue_comments", "releases"]
WINSOR_Q = 0.99
KEYS = ["repo_name", "season"]
OBSERVATION_WINDOWS_DAYS = (30, 90)


def winsor_upper(s, q=WINSOR_Q):
    # Float64: quantile can be non-integer; Int64 cannot store the cap exactly
    x = s.astype("float64")
    return x.clip(lower=0, upper=x.quantile(q))


prep_activity = prep_activity.copy()
for col in ACTIVITY_COUNT_COLS:
    if col in prep_activity.columns:
        prep_activity[col + "_winsor"] = winsor_upper(prep_activity[col])

winsor_cols = [c + "_winsor" for c in ACTIVITY_COUNT_COLS if c + "_winsor" in prep_activity.columns]
prep_activity[winsor_cols].describe().T[["max"]]


,max
watches_winsor,1536.02
pushes_winsor,1192.02
issues_winsor,301.00
issue_comments_winsor,1169.01
releases_winsor,33.00


The table above is the per-column `max` from `describe()` on each `*_winsor` field—i.e. the largest value still present after capping at the 99th percentile. It is a quick sanity check that `clip` behaved as expected.

## **Construct Data**


### Fork–PR conversion labels

The dataset does not link each fork event to a pull request id, so we need an operational definition of conversion. We use a time window and a shared identity:

- A fork converts if there is at least one PR by the same `user_login`, with fork time *strictly before* PR time, and the PR time within `observation_window_days` after that fork.  
- We count per fork; `fork_conversion_rate` is the share of fork events in the list that convert, *not* “all PR events divided by fork count” in one ratio.

The next cells define three helpers: `as_event_list` (parquet cell -> list of events), `count_converted_forks` (match PRs to forks in a time window), and `build_labels` (one row per `(repo_name, season)`).

We build labels for 30 and 90 days and save two parquet files (`labels_30d` and `labels_90d`).


#### `as_event_list`
Parquet cell (forks or prs) -> list of event dicts.


In [15]:
def as_event_list(cell):
    if cell is None or (isinstance(cell, float) and np.isnan(cell)):
        return []
    arr = np.asarray(cell, dtype=object)
    if arr.size == 0:
        return []
    out = []
    for x in arr.ravel():
        if x is None or x is pd.NA:
            continue
        if isinstance(x, dict):
            out.append(x)
            continue
        try:
            names = getattr(x.dtype, "names", None)
            if names:
                out.append({k: x[k] for k in names})
                continue
        except Exception:
            pass
        try:
            item = x.item()
            if isinstance(item, dict):
                out.append(item)
        except Exception:
            pass
    return out


#### `count_converted_forks`
Counts forks with a same-user PR in the observation window after fork time.


In [16]:
def count_converted_forks(fork_events, pr_events, obs_days):
    if not fork_events:
        return 0, 0
    pr_by_user = {}
    for pr in pr_events:
        login = pr.get("user_login")
        if login is None:
            continue
        pr_by_user.setdefault(login, []).append(pd.Timestamp(pr["event_time"]))
    for ts_list in pr_by_user.values():
        ts_list.sort()
    n_forks = len(fork_events)
    converted = 0
    window = pd.Timedelta(days=obs_days)
    for fk in fork_events:
        login = fk.get("user_login")
        if login is None:
            continue
        t0 = pd.Timestamp(fk["event_time"])
        upper = t0 + window
        for pr_t in pr_by_user.get(login, []):
            if t0 < pr_t <= upper:
                converted += 1
                break
    return n_forks, converted


#### `build_labels`
Builds a label DataFrame for one `obs_days` from `fork_pr`.


In [17]:
def build_labels(fork_pr_df, obs_days):
    rows = []
    for row in fork_pr_df.itertuples(index=False):
        forks = as_event_list(row.forks)
        prs = as_event_list(row.prs)
        nf, nc = count_converted_forks(forks, prs, obs_days)
        rate = nc / nf if nf else np.nan
        rows.append(
            {
                "repo_name": row.repo_name,
                "season": row.season,
                "observation_window_days": obs_days,
                "n_forks": nf,
                "n_converted_forks": nc,
                "fork_conversion_rate": rate,
                "n_pr_events": len(prs),
            }
        )
    return pd.DataFrame(rows)


### Build label tables and save

We materialize both windows, save two parquet files, and print `describe()` on the same label columns for 30d and 90d.


In [18]:
from pathlib import Path
Path("data").mkdir(parents=True, exist_ok=True)
label_paths = {
    30: "data/fork_conversion_labels_obs30d.parquet",
    90: "data/fork_conversion_labels_obs90d.parquet",
}
labels_by_window = {}
for d in OBSERVATION_WINDOWS_DAYS:
    labels_by_window[d] = build_labels(fork_pr, d)
    path = label_paths[d]
    try:
        labels_by_window[d].to_parquet(path, index=False)
        print("Saved", path, labels_by_window[d].shape)
    except ImportError:
        csv_path = path.replace(".parquet", ".csv")
        labels_by_window[d].to_csv(csv_path, index=False)
        print("Saved", csv_path, labels_by_window[d].shape)

labels_30d = labels_by_window[30]
labels_90d = labels_by_window[90]
_cols = ["n_forks", "n_converted_forks", "fork_conversion_rate"]
print("describe label cols — 30d:")
print(labels_30d[_cols].describe())
print("describe label cols — 90d:")
print(labels_90d[_cols].describe())


Saved data/fork_conversion_labels_obs30d.parquet (100000, 7)
Saved data/fork_conversion_labels_obs90d.parquet (100000, 7)
describe label cols — 30d:
             n_forks  n_converted_forks  fork_conversion_rate
count  100000.000000      100000.000000         100000.000000
mean       29.819910           2.912260              0.128063
std       156.692666          20.424703              0.216470
min         6.000000           0.000000              0.000000
25%         7.000000           0.000000              0.000000
50%        11.000000           0.000000              0.000000
75%        20.000000           2.000000              0.166667
max     14513.000000        3383.000000              1.000000
describe label cols — 90d:
             n_forks  n_converted_forks  fork_conversion_rate
count  100000.000000      100000.000000         100000.000000
mean       29.819910           3.020300              0.132753
std       156.692666          20.679535              0.221853
min         6.0000

### Sensitivity: 30d vs 90d

We compare how stable repository-level ranks of `fork_conversion_rate` are when the label window changes. The next cell prints summary stats, a blank line, then a MODELING block that says which `data/modeling_dataset_obs{30,90}d.parquet` to use first (or to fit both).

What does each line mean:

- `n with forks>0 in both windows` — how many `repo_name`×`season` rows are used (both label builds need forks).  
- `Spearman rho` / `Kendall tau` — agreement of ranks between 30d and 90d rates (≈1 means almost the same ordering of repos).  
- `Decile reversals` — stricter check: how often a repo is in the *top* decile in one window and the *bottom* in the other (large = unstable ranks).  
- `Conversions in 90d but not 30d` — of all 90d conversion *events* (scaled counts), what fraction are added only by extending 30d→90d (i.e. PRs after day 30 in that window). Small → 30d captures most conversion mass.  
- `Median (rate_90d - rate_30d)` — typical shift in the *rate* when switching window length.

Turning the numbers into a modeling choice:

- High rank agreement, few reversals, and small `Conversions in 90d but not 30d`: a common *modeling* starting point is `data/modeling_dataset_obs30d.parquet`; re-run (or test robustness) with `...obs90d` if the task must value PRs in days 31–90. (This is guidance for Modeling, not a setting inside this notebook.)  
- Low Spearman, many decile reversals, or a large 31–90d share: do not treat the two windows as interchangeable; prefer `...obs90d` for the target if the business window is 90d, and/or build two models, one per file.  
- After a blank line, the next block is a MODELING recommendation: which `data/modeling_dataset_obs{30,90}d.parquet` to load first, or to fit both, matching the heuristics above.


In [19]:
_key = ["repo_name", "season"]
cmp = (
    labels_30d[_key + ["n_forks", "n_converted_forks", "fork_conversion_rate"]]
    .rename(
        columns={
            "n_forks": "n_forks_30",
            "n_converted_forks": "nc_30",
            "fork_conversion_rate": "rate_30d",
        }
    )
    .merge(
        labels_90d[_key + ["n_forks", "n_converted_forks", "fork_conversion_rate"]]
        .rename(
            columns={
                "n_forks": "n_forks_90",
                "n_converted_forks": "nc_90",
                "fork_conversion_rate": "rate_90d",
            }
        ),
        on=_key,
        how="inner",
        validate="one_to_one",
    )
)
mask = (cmp["n_forks_30"] > 0) & (cmp["n_forks_90"] > 0)
sub = cmp.loc[mask, ["rate_30d", "rate_90d"]].astype(float)
n_ok = len(sub)
rho = sub["rate_30d"].corr(sub["rate_90d"], method="spearman")
try:
    tau = sub["rate_30d"].corr(sub["rate_90d"], method="kendall")
except Exception:
    tau = float("nan")
p30 = sub["rate_30d"].rank(pct=True, method="average")
p90 = sub["rate_90d"].rank(pct=True, method="average")
strong_reversal = ((p30 >= 0.9) & (p90 <= 0.1)) | ((p30 <= 0.1) & (p90 >= 0.9))
n_rev = int(strong_reversal.sum())
frac_rev = n_rev / n_ok if n_ok else np.nan
nc30_s = cmp.loc[mask, "nc_30"].astype(int)
nc90_s = cmp.loc[mask, "nc_90"].astype(int)
extra_conv = (nc90_s - nc30_s).clip(lower=0)
tot90 = int(nc90_s.sum())
frac_extra = (float(extra_conv.sum()) / tot90) if tot90 > 0 else float("nan")
median_drate = float((sub["rate_90d"] - sub["rate_30d"]).median())
print("Sensitivity (n with forks>0 in both windows: {})".format(n_ok))
print("  Spearman rho: {:.4f}   Kendall tau: {:.4f}".format(rho, tau))
print("  Decile reversals: {} ({:.2%})".format(n_rev, frac_rev))
print("  Conversions in 90d but not 30d: {:.2%} of all 90d conversions".format(frac_extra))
print("  Median (rate_90d - rate_30d): {:.4f}".format(median_drate))
if n_ok < 2 or not np.isfinite(rho):
    verdict = (
        "Too few rows of comparable forks for a rank comparison, or Spearman is undefined.\n"
        "MODELING: do not use this check alone. Pick 30d vs 90d from the problem statement; "
        "then load the matching `data/modeling_dataset_obs30d.parquet` or `data/modeling_dataset_obs90d.parquet`."
    )
elif (rho < 0.65) or (frac_rev > 0.05):
    verdict = (
        "Ranks of fork_conversion_rate at 30d vs 90d disagree (Spearman rho={:.2f}, decile reversals {:.1%} of rows).\n"
        "MODELING: the two label definitions are not interchangeable. Train at least two baselines: "
        "`data/modeling_dataset_obs30d.parquet` and `data/modeling_dataset_obs90d.parquet`, "
        "then keep the file whose `observation_window_days` match your evaluation plan."
    ).format(rho, frac_rev)
elif (rho >= 0.75) and (frac_extra <= 0.18) and (frac_rev <= 0.02):
    verdict = (
        "Ranks of conversion rate match (Spearman rho={:.2f}). "
        "Only {:.1%} of all 90d conversion events are missing at 30d (PRs after day 30).\n"
        "MODELING: use `data/modeling_dataset_obs30d.parquet` for the main model. "
        "Optionally re-run on `data/modeling_dataset_obs90d.parquet` to stress the label, "
        "or as the only table if 31-90d PRs must be part of the target."
    ).format(rho, frac_extra)
elif (rho >= 0.75) and (frac_extra > 0.18):
    verdict = (
        "A large share of 90d conversion events is missing from 30d counts ({:.1%} of all 90d events are added only in days 31-90).\n"
        "MODELING: use `data/modeling_dataset_obs90d.parquet` as the main label file; a 30d target would under-count."
    ).format(frac_extra)
else:
    pick = 90 if frac_extra > 0.12 else 30
    verdict = (
        "Borderline agreement (Spearman rho={:.2f}; extra 90d events not in 30d: {:.1%} of 90d total).\n"
        "MODELING: start with `data/modeling_dataset_obs{2}d.parquet` and compare to the other window."
    ).format(rho, frac_extra, pick)
print()
print(verdict)


Sensitivity (n with forks>0 in both windows: 100000)
  Spearman rho: 0.9910   Kendall tau: 0.9805
  Decile reversals: 0 (0.00%)
  Conversions in 90d but not 30d: 3.58% of all 90d conversions
  Median (rate_90d - rate_30d): 0.0000

Ranks of conversion rate match (Spearman rho=0.99). Only 3.6% of all 90d conversion events are missing at 30d (PRs after day 30).
MODELING: use `data/modeling_dataset_obs30d.parquet` for the main model. Optionally re-run on `data/modeling_dataset_obs90d.parquet` to stress the label, or as the only table if 31-90d PRs must be part of the target.


### Logarithmic features (`log1p`)

We add `log1p` on each raw count and on each winsorized count, following the EDA conclusion that distributions are skewed. Modeling can later pick raw- or winsor-based `log1p` features and drop duplicates as needed.


In [20]:
for col in ACTIVITY_COUNT_COLS:
    if col in prep_activity.columns:
        prep_activity["log1p_" + col] = np.log1p(prep_activity[col].astype(float))
    wc = col + "_winsor"
    if wc in prep_activity.columns:
        prep_activity["log1p_" + wc] = np.log1p(prep_activity[wc].astype(float))

sorted([c for c in prep_activity.columns if c.startswith("log1p_")])


['log1p_issue_comments',
 'log1p_issue_comments_winsor',
 'log1p_issues',
 'log1p_issues_winsor',
 'log1p_pushes',
 'log1p_pushes_winsor',
 'log1p_releases',
 'log1p_releases_winsor',
 'log1p_watches',
 'log1p_watches_winsor']

## **Integrate Data**

We inner-join `prep_activity` to each label table (`labels_30d` and `labels_90d`) on `repo_name` and `season`, with a one-to-one key check.


In [21]:
# Inner join: one merged frame per observation window (same keys, different label columns)
modeling_df_30d = prep_activity.merge(labels_30d, on=KEYS, how="inner", validate="one_to_one")
modeling_df_90d = prep_activity.merge(labels_90d, on=KEYS, how="inner", validate="one_to_one")

assert modeling_df_30d.shape[0] == prep_activity.shape[0] == labels_30d.shape[0]
assert modeling_df_90d.shape[0] == prep_activity.shape[0] == labels_90d.shape[0]
print("modeling_df_30d:", modeling_df_30d.shape, "  modeling_df_90d:", modeling_df_90d.shape)
print("\nSample — 30d labels merged (first 5 rows):")
print(modeling_df_30d.head(5).to_string())
print("\nSample — 90d labels merged (first 5 rows):")
print(modeling_df_90d.head(5).to_string())


modeling_df_30d: (100000, 27)   modeling_df_90d: (100000, 27)

Sample — 30d labels merged (first 5 rows):
                               repo_name         season  watches  pushes  issues  issue_comments  releases  watches_winsor  pushes_winsor  issues_winsor  issue_comments_winsor  releases_winsor  log1p_watches  log1p_watches_winsor  log1p_pushes  log1p_pushes_winsor  log1p_issues  log1p_issues_winsor  log1p_issue_comments  log1p_issue_comments_winsor  log1p_releases  log1p_releases_winsor  observation_window_days  n_forks  n_converted_forks  fork_conversion_rate  n_pr_events
0  GoodRequest/BackendAssignment-Fitness  winter_spring        0       1       0               0         0             0.0            1.0            0.0                    0.0              0.0            0.0                   0.0      0.693147             0.693147           0.0                  0.0                   0.0                          0.0             0.0                    0.0                       30  

## **Format Data**


### Export for Modeling

We add:

- `repo_id` — integer code for `repo_name` (convenience for tools that want numeric ids)  
- `has_conversion` — 1 if `n_converted_forks > 0`  
- `activity_strength` — sum of the five `log1p_*` on raw counts (single summary; optional for analysis or `stratify`)  
- `stratum_activity_quintile`, `stratum_fork_quintile` — quintile bins (0..4) for optional stratified splits; not all need to be used as model features

We then shuffle rows with a fixed seed and write `modeling_dataset_obs30d.parquet` and `modeling_dataset_obs90d.parquet` (one per observation window).

#### `_qcut_quintile`

Map a numeric series to quintile labels (0..4) with `pd.qcut` (`duplicates="drop"`). Used for `stratum_*_quintile` only; on degenerate input returns all zeros.

In [22]:
def _qcut_quintile(s):
    s = pd.to_numeric(s, errors="coerce")
    if s.notna().sum() == 0 or s.nunique(dropna=True) < 2:
        return pd.Series(0, index=s.index, dtype="Int64")
    try:
        b = pd.qcut(s, q=5, labels=False, duplicates="drop")
    except ValueError:
        return pd.Series(0, index=s.index, dtype="Int64")
    return b.astype("Int64")

In [23]:
def _merge_and_ids(lab_df):
    m = prep_activity.merge(lab_df, on=KEYS, how="inner", validate="one_to_one")
    assert m.shape[0] == prep_activity.shape[0] == lab_df.shape[0]
    m = m.copy()
    m["repo_id"] = pd.Categorical(m["repo_name"]).codes
    m["has_conversion"] = (m["n_converted_forks"] > 0).astype(int)
    log1p_base = [f"log1p_{c}" for c in ACTIVITY_COUNT_COLS if f"log1p_{c}" in m.columns]
    m["activity_strength"] = m[log1p_base].sum(axis=1)
    m["stratum_activity_quintile"] = _qcut_quintile(m["activity_strength"])
    m["stratum_fork_quintile"] = _qcut_quintile(m["n_forks"].astype("float64"))
    return m


def _to_final(m):
    return m[META + FEAT + STRAT + LBL + REST].sample(frac=1.0, random_state=335).reset_index(drop=True)


def _save_df(path_base, df):
    try:
        df.to_parquet(path_base + ".parquet", index=False)
        print("Saved", path_base + ".parquet", df.shape)
    except ImportError:
        df.to_csv(path_base + ".csv", index=False)
        print("Saved", path_base + ".csv (install pyarrow for parquet)", df.shape)


# Column layout is the same for either window; pick one to discover FEAT/REST (here 30d).
_template = _merge_and_ids(labels_30d)
META = ["repo_name", "repo_id", "season"]
FEAT = sorted([c for c in _template.columns if c.startswith("log1p_")])
STRAT = ["activity_strength", "stratum_activity_quintile", "stratum_fork_quintile"]
LBL = [
    "observation_window_days",
    "n_forks",
    "n_converted_forks",
    "fork_conversion_rate",
    "n_pr_events",
    "has_conversion",
]
REST = [c for c in _template.columns if c not in META + STRAT + FEAT + LBL]
for _d, _lab in ((30, labels_30d), (90, labels_90d)):
    base = "data/modeling_dataset_obs{}d".format(_d)
    _save_df(base, _to_final(_merge_and_ids(_lab)))

P30, P90 = "data/modeling_dataset_obs30d.parquet", "data/modeling_dataset_obs90d.parquet"
d30 = pd.read_parquet(P30)
d90 = pd.read_parquet(P90)
assert len(d30) == len(d90)
peek = [
    "repo_name",
    "season",
    "observation_window_days",
    "n_forks",
    "n_converted_forks",
    "fork_conversion_rate",
    "has_conversion",
]
print()
print("Peek: first 3 rows (30d, shuffled on save):")
print(d30[peek].head(3).to_string())
print()
print("Label column means (numeric) — 30d vs 90d:")
print(
    pd.DataFrame({"30d": d30[LBL].mean(numeric_only=True), "90d": d90[LBL].mean(numeric_only=True)}).round(4)
)


Saved data/modeling_dataset_obs30d.parquet (100000, 32)
Saved data/modeling_dataset_obs90d.parquet (100000, 32)

Peek: first 3 rows (30d, shuffled on save):
                                        repo_name         season  observation_window_days  n_forks  n_converted_forks  fork_conversion_rate  has_conversion
0  AutomationPanda/awesome-web-testing-playwright  winter_spring                       30        8                  0                   0.0               0
1                                     Neamar/KISS  summer_autumn                       30       10                  3                   0.3               1
2                          magnific0/wondershaper  winter_spring                       30        6                  0                   0.0               0

Label column means (numeric) — 30d vs 90d:
                             30d      90d
observation_window_days  30.0000  90.0000
n_forks                  29.8199  29.8199
n_converted_forks         2.9123   3.0203
fork_co

### What the lines above show

`Saved data/modeling_dataset_obs{30,90}d.parquet (N, 32)` — Confirms each Parquet file was written. `N` is the number of rows (one per `repo_name` × `season`); `32` is the full column set: activity features, optional strata, label fields, and raw/winsor counts for that observation window.

`Peek: first 3 rows (30d, shuffled on save)` — Read back from the 30d modeling file only. Row order is random (shuffle on save with a fixed seed), so this is a quick sanity sample, not the original extract order. Columns: keys, `observation_window_days`, and core fork–PR label fields.

`Label column means` (30d vs 90d) — Column means of the numeric label fields, side by side. Expect:
- `observation_window_days` = 30 vs 90 (by construction).
- `n_forks` and `n_pr_events` the same in both columns (same underlying fork/PR lists; the window only changes which PRs count as *conversions*).
- `n_converted_forks`, `fork_conversion_rate`, `has_conversion` often a bit higher at 90d when extra PRs appear after day 30. If the gaps are small, aggregate label behavior is close between windows.


Train/test split is not done here; it is part of the Modeling notebook.

Handoff to Modeling: use `data/modeling_dataset_obs30d.parquet` or `data/modeling_dataset_obs90d.parquet` as the main input. They share the same features and metadata; only the label side differs (30- vs 90-day conversion). The `fork_conversion_labels_obs*.parquet` files are for checks or reproducing labels; for model fitting, load a modeling parquet, pick feature columns (primarily `log1p_*`, optionally strata), and the label fields you need (`fork_conversion_rate`, `n_converted_forks`, or `has_conversion`, depending on the task).
